# Vérification des chiffres clés en SQL (DuckDB)

**Objectif** : recalculer en SQL les indicateurs clés obtenus en Python (notebook 02), afin de garantir leur exactitude avant leur diffusion.

## 1. Chargement des données

In [3]:
import duckdb

con = duckdb.connect()

# Table "clients" : lecture du fichier + indicateur de départ (1 = parti, 0 = actif)
con.sql("""
CREATE OR REPLACE TABLE clients AS
SELECT *,
       CASE WHEN Attrition_Flag = 'Attrited Customer' THEN 1 ELSE 0 END AS parti
FROM read_csv_auto('../data/BankChurners.csv')
""")

## 2. Indicateurs globaux

In [4]:
con.sql("""
SELECT COUNT(*)                   AS nb_clients,
       SUM(parti)::INT            AS nb_partis,
       ROUND(AVG(parti) * 100, 1) AS taux_depart_pct
FROM clients
""").df()

,nb_clients,nb_partis,taux_depart_pct
0,10127,1627,16.1


## 3. Grille des 6 signaux d'alerte

In [5]:
# Vue "signaux" : nombre de signaux d'alerte par client (mêmes règles que le notebook 02)
con.sql("""
CREATE OR REPLACE VIEW signaux AS
WITH seuils AS (
    SELECT QUANTILE_CONT(Total_Trans_Ct, 0.25)      AS seuil_transactions,
           QUANTILE_CONT(Total_Ct_Chng_Q4_Q1, 0.25) AS seuil_evolution
    FROM clients
)
SELECT c.CLIENTNUM,
       c.parti,
         (c.Total_Trans_Ct <= s.seuil_transactions)::INT
       + (c.Total_Ct_Chng_Q4_Q1 <= s.seuil_evolution)::INT
       + (c.Total_Revolving_Bal = 0)::INT
       + (c.Total_Relationship_Count <= 2)::INT
       + (c.Months_Inactive_12_mon >= 3)::INT
       + (c.Contacts_Count_12_mon >= 4)::INT AS nb_signaux
FROM clients AS c
CROSS JOIN seuils AS s
""")

con.sql("""
SELECT nb_signaux,
       COUNT(*)             AS nb_clients,
       ROUND(AVG(parti), 3) AS taux_depart
FROM signaux
GROUP BY nb_signaux
ORDER BY nb_signaux
""").df()

,nb_signaux,nb_clients,taux_depart
0,0,1696,0.017
1,1,3654,0.043
2,2,2877,0.139
3,3,1243,0.398
4,4,473,0.795
5,5,157,0.917
6,6,27,1.000


## 4. Cible prioritaire

In [6]:
con.sql("""
SELECT COUNT(*) FILTER (WHERE nb_signaux >= 3)                                  AS clients_cibles,
       ROUND(AVG(parti) FILTER (WHERE nb_signaux >= 3) * 100, 1)                AS taux_depart_cible_pct,
       ROUND(SUM(parti) FILTER (WHERE nb_signaux >= 3) * 100.0 / SUM(parti), 1) AS part_departs_couverts_pct
FROM signaux
""").df()

,clients_cibles,taux_depart_cible_pct,part_departs_couverts_pct
0,1900,54.8,64.0


## 5. Conclusion : contrôle de cohérence Python / SQL

| Indicateur | Python | SQL | Écart |
|---|---|---|---|
| Nombre de clients | 10 127 | 10 127 | Aucun |
| Clients partis | 1 627 | 1 627 | Aucun |
| Taux de départ de référence | 16,1 % | 16,1 % | Aucun |
| Clients ciblés (3 signaux ou plus) | 1 900 | 1 900 | Aucun |
| Taux de départ dans la cible | 54,8 % | 54,8 % | Aucun |
| Part des départs couverts par la cible | 64,0 % | 64,0 % | Aucun |

**En résumé** : les indicateurs clés sont identiques dans les deux outils. Le constat central de l'analyse est confirmé : **en ciblant 1 900 clients (moins d'1 client sur 5), la banque peut toucher près des 2/3 des départs.** Les chiffres présentés dans le tableau de bord reposent sur des calculs vérifiés.